In [3]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import time
from torch.utils.data import DataLoader
from torchvision import transforms, datasets

In [4]:
def nin_block(in_channels, out_channels, kernel_size, stride, padding):
    blk = nn.Sequential(nn.Conv2d(in_channels, out_channels, kernel_size, stride, padding),
                        nn.ReLU(),
                        nn.Conv2d(out_channels, out_channels, kernel_size=1),
                        nn.ReLU(),
                        nn.Conv2d(out_channels, out_channels, kernel_size=1),
                        nn.ReLU())
    return blk


In [5]:
class GlobalAvgPool2d(nn.Module):
    # 全局平均池化层可通过将池化层窗口设置成输入的高和宽实现
    def __init__(self):
        super(GlobalAvgPool2d, self).__init__()
    def forward(self, x):
        return F.avg_pool2d(x, kernel_size=x.size()[2:])

net = nn.Sequential(
    nin_block(3,96,kernel_size=11,stride=4,padding=0),
    nn.MaxPool2d(kernel_size=2,stride=2),
    nin_block(96,256,kernel_size=5,stride=1,padding=2),
    nn.MaxPool2d(kernel_size=2,stride=2),
    nin_block(256,384,kernel_size=3,stride=1,padding=1),
    nn.MaxPool2d(kernel_size=2,stride=2),
    nin_block(384,10,kernel_size=3,stride=1,padding=1),
    GlobalAvgPool2d(),
    nn.Flatten()
)

In [6]:
x = torch.rand(1,3,224,224)
# net(x).shape  # 测试形状，已包含在后续遍历中

for name, blk in net.named_children():
    x = blk(x)
    print(name, 'output size', x.shape)

0 output size torch.Size([1, 96, 54, 54])
1 output size torch.Size([1, 96, 27, 27])
2 output size torch.Size([1, 256, 27, 27])
3 output size torch.Size([1, 256, 13, 13])
4 output size torch.Size([1, 384, 13, 13])
5 output size torch.Size([1, 384, 6, 6])
6 output size torch.Size([1, 10, 6, 6])
7 output size torch.Size([1, 10, 1, 1])
8 output size torch.Size([1, 10])


In [7]:
data_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

train_sets = datasets.CIFAR10(root='cifar_10', train=True, download=True, transform=data_transform)
test_sets = datasets.CIFAR10(root='cifar_10', train=False, download=True, transform=data_transform)
batch_size = 64
train_loader = torch.utils.data.DataLoader(dataset=train_sets, batch_size=batch_size, shuffle=True)
test_loader = torch.utils.data.DataLoader(dataset=test_sets, batch_size=batch_size, shuffle=True)


In [13]:
def evaluate_accuracy(data_iter, net):
    acc_sum, n = 0.0, 0
    net.eval()
    for X, y in data_iter:
        X = X.cuda()  # 注意：这里若设备不是 CUDA 会出错，更严谨的是用 to(device)，但按用户代码先保留
        y = y.cuda()
        acc_sum += (net(X).argmax(dim=1) == y).float().sum().item()
        n += y.shape[0]
    return acc_sum / n

In [15]:
def train(net, train_iter, test_iter, batch_size, optimizer, device, num_epochs):
    net = net.to(device)
    print("training on ", device)
    loss = torch.nn.CrossEntropyLoss()
    batch_count = 0
    for epoch in range(num_epochs):
        train_l_sum, train_acc_sum, n, start = 0.0, 0.0, 0, time.time()
        net.train()
        for X, y in train_iter:
            X = X.to(device)
            y = y.to(device)
            y_hat = net(X)
            l = loss(y_hat, y)
            optimizer.zero_grad()
            l.backward()
            optimizer.step()
            train_l_sum += l.cpu().item()
            train_acc_sum += (y_hat.argmax(dim=1) == y).sum().cpu().item()
            n += y.shape[0]
            batch_count += 1
        test_acc = evaluate_accuracy(test_iter, net)
        print('epoch %d, loss %.4f, train acc %.3f, test acc %.3f, time %.1f sec'
              % (epoch + 1, train_l_sum / batch_count, train_acc_sum / n, test_acc, time.time() - start))


In [17]:
train_iter, test_iter = train_loader, test_loader
lr = 0.0001
num_epochs = 5
optimizer = torch.optim.Adam(net.parameters(), lr)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
train(net, train_iter, test_iter, batch_size, optimizer, device, num_epochs)

training on  cuda
epoch 1, loss 2.3044, train acc 0.100, test acc 0.100, time 91.6 sec
epoch 2, loss 1.1517, train acc 0.100, test acc 0.100, time 91.8 sec
epoch 3, loss 0.7677, train acc 0.100, test acc 0.100, time 91.5 sec
epoch 4, loss 0.5757, train acc 0.100, test acc 0.100, time 89.0 sec
epoch 5, loss 0.4605, train acc 0.100, test acc 0.100, time 90.5 sec
